# AI Agent with Lakebase Memory — DatabricksOpenAI

The native-Databricks version of the LLM demo: same Lakebase Postgres session memory and the same OpenAI Agents SDK loop, but the model client is now **`DatabricksOpenAI`** (from the `databricks-openai` package) — Databricks' drop-in replacement for the OpenAI client. No token minting, no base-URL wiring: unified auth talks to **Unity Gateway** for you. Model: `system.ai.llama-4-maverick` (low-cost, governed). Runs on serverless compute, Python only.

## What changes vs. the manual OpenAI client version

In *Lakebase Agent Memory - OpenAI Agents SDK*, we wired the model call by hand: mint a workspace token, build an `AsyncOpenAI` client, point it at `<workspace>/ai-gateway/mlflow/v1`, and remember to refresh the token hourly.

This notebook swaps that for **`DatabricksOpenAI`**, Databricks' official drop-in replacement for the OpenAI client:

* **Zero auth code** — it reuses the Databricks SDK's unified authentication (your notebook session, a service principal in an app, OAuth in CI — whatever the environment provides). No tokens minted, nothing to refresh.
* **One flag for the Gateway** — `use_ai_gateway=True` routes calls through Unity Gateway (`/ai-gateway/mlflow/v1`) so you keep governed `system.ai.*` model names, UC permissions, and usage tracking.
* **⚠️ The flag matters** — without it, the client defaults to the *classic Model Serving* endpoint URL (`/serving-endpoints`), and governed UC names like `system.ai.llama-4-maverick` fail with `ENDPOINT_NOT_FOUND`. This is a great gotcha to teach.
* **Bonus capabilities** — the same package also ships Databricks-native agent tools (`UCFunctionToolkit` to call Unity Catalog functions, `VectorSearchRetrieverTool`, `McpServerToolkit`), so this client is the natural on-ramp to tool-using agents on Databricks.

The memory loop is unchanged: retrieve history from Lakebase, pass it to the LLM as context, persist both sides of the conversation back to Lakebase. The punchline is the same: a brand-new agent instance still knows who you are — memory lives in Postgres, not the process or the model.

In [0]:
%pip install databricks-openai openai-agents pg8000
# databricks-openai: DatabricksOpenAI / AsyncDatabricksOpenAI clients + native agent tools.
# openai-agents: the OpenAI Agents SDK. pg8000: pure-Python Postgres driver.
# NOTE: do NOT install psycopg/psycopg2 here; they crash the serverless kernel.

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 35.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 70.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 176.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 146.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 94.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 143.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 75.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 106.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 137.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.1/811.1 kB 44.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 789.0/789.0 kB 44.8 MB/s  0:00:00
  Attempting uninstall: wcwidth
    Found existing installation: wcwidth 0.2.13
    Not uninstalling wcwidth at /databricks/python3/lib/pyth

In [0]:
# Lakebase connection helper — IDENTICAL construct to the other demos.
# Fresh connection per operation, OAuth token minted from the Databricks
# identity each time, no passwords anywhere.
import ssl

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()  # pre-authenticated with your Databricks identity

LAKEBASE_HOST = "ep-cold-forest-d8ignwgw.database.us-east-2.cloud.databricks.com"
LAKEBASE_DATABASE = "databricks_postgres"
LAKEBASE_ENDPOINT = "projects/alan/branches/production/endpoints/primary"


def get_connection():
    """Open a fresh connection to the Lakebase database (pg8000 driver)."""
    import pg8000.native

    token = w.postgres.generate_database_credential(
        endpoint=LAKEBASE_ENDPOINT
    ).token  # NEVER print or log this token

    ctx = ssl.create_default_context()  # equivalent of sslmode=require
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE

    return pg8000.native.Connection(
        user=w.current_user.me().user_name,
        host=LAKEBASE_HOST,
        database=LAKEBASE_DATABASE,
        password=token,
        ssl_context=ctx,
        timeout=30,
    )


# Smoke test
conn = get_connection()
rows = conn.run("SELECT current_database(), current_user, version()")
conn.close()
print("Connected to:", rows[0][0], "| logged in as:", rows[0][1])

Connected to: databricks_postgres | logged in as: alan.dennis@databricks.com


In [0]:
# DatabricksOpenAI setup — the whole Gateway wiring, replaced by one constructor.
#
# AsyncDatabricksOpenAI is the async variant (the Agents SDK Runner is async).
#
# use_ai_gateway=True  -> routes to Unity Gateway (/ai-gateway/mlflow/v1),
#                         which understands governed UC names like system.ai.*.
# Without it, the client defaults to classic Model Serving (/serving-endpoints)
# and system.ai.* names fail with ENDPOINT_NOT_FOUND.
#
# Authentication: handled by Databricks unified auth (notebook session here,
# service principal in an App, OAuth in CI). No token minting in our code.
from agents import set_tracing_disabled
from databricks_openai import AsyncDatabricksOpenAI

MODEL = "system.ai.llama-4-maverick"  # low-cost, open-weights, governed UC model service

gw_client = AsyncDatabricksOpenAI(use_ai_gateway=True)
print("Client base URL:", gw_client.base_url)

set_tracing_disabled(True)  # Agents SDK tracing to the OpenAI platform is not available here
print("DatabricksOpenAI client ready (auto-authenticated, Gateway-routed).")

Client base URL: https://dbc-6257a4e4-ffad.cloud.databricks.com/ai-gateway/mlflow/v1/
DatabricksOpenAI client ready (auto-authenticated, Gateway-routed).


In [0]:
# Smoke test: one agent run through the Gateway before we build the demo.
from agents import Agent, ModelSettings, OpenAIChatCompletionsModel, Runner

probe_agent = Agent(
    name="ProbeAgent",
    instructions="You are a terse assistant. Reply with exactly what is asked.",
    model=OpenAIChatCompletionsModel(model=MODEL, openai_client=gw_client),
    model_settings=ModelSettings(max_tokens=32),
)

# The Agents SDK Runner is async — notebooks (IPython) support top-level await.
probe_result = await Runner.run(probe_agent, "Reply with exactly: OK")
print("Gateway + DatabricksOpenAI smoke test:", probe_result.final_output)

Gateway + DatabricksOpenAI smoke test: OK


In [0]:
# Memory schema — same design, its OWN table so the three demo notebooks
# don't wipe each other's data when re-run. DROP first for re-runnability.
conn = get_connection()
conn.run("DROP TABLE IF EXISTS agent_session_memory_databricks")
conn.run(
    """
    CREATE TABLE agent_session_memory_databricks (
        session_id  TEXT        NOT NULL,
        turn        INTEGER     NOT NULL,
        role        TEXT        NOT NULL,   -- 'user' or 'agent'
        content     TEXT        NOT NULL,
        created_at  TIMESTAMPTZ DEFAULT now(),
        PRIMARY KEY (session_id, turn)
    )
    """
)
conn.close()
print("Table agent_session_memory_databricks created.")

Table agent_session_memory_databricks created.


In [0]:
# SessionMemory — IDENTICAL construct to the other demos (only the table differs).
class SessionMemory:
    """Persistent conversation memory backed by a Lakebase Postgres table.

    Every method opens a fresh connection (minting a fresh OAuth token),
    so it works no matter which process — or which replica — calls it.
    """

    def __init__(self, table):
        self.table = table

    def save_turn(self, session_id, turn, role, content):
        """INSERT one turn of the conversation."""
        conn = get_connection()
        conn.run(
            f"INSERT INTO {self.table} (session_id, turn, role, content) "
            "VALUES (:session_id, :turn, :role, :content)",
            session_id=session_id,
            turn=turn,
            role=role,
            content=content,
        )
        conn.close()

    def get_history(self, session_id):
        """SELECT all turns for a session, ordered by turn."""
        conn = get_connection()
        rows = conn.run(
            f"SELECT turn, role, content FROM {self.table} "
            "WHERE session_id = :session_id ORDER BY turn",
            session_id=session_id,
        )
        conn.close()
        return [{"turn": t, "role": r, "content": c} for (t, r, c) in rows]

    def forget_session(self, session_id):
        """DELETE every turn for a session (a full memory wipe)."""
        conn = get_connection()
        conn.run(
            f"DELETE FROM {self.table} WHERE session_id = :session_id",
            session_id=session_id,
        )
        conn.close()


memory = SessionMemory(table="agent_session_memory_databricks")
print("SessionMemory ready (table: agent_session_memory_databricks).")

SessionMemory ready (table: agent_session_memory_databricks).


In [0]:
# The agent — same three-step loop, reply via the OpenAI Agents SDK running
# on the DatabricksOpenAI client.
from agents import Agent, ModelSettings, OpenAIChatCompletionsModel, Runner


llm_agent = Agent(
    name="MemoryAgent",
    instructions=(
        "You are a friendly assistant in a live demo. You have persistent "
        "memory: the conversation so far is given to you as context. Keep "
        "replies to one or two sentences. When the user asks what their name "
        "is or what they like, answer from the conversation context."
    ),
    model=OpenAIChatCompletionsModel(model=MODEL, openai_client=gw_client),
    model_settings=ModelSettings(max_tokens=200),
)


class DemoAgent:
    """Agent with Lakebase-backed session memory and an LLM reply step.

    On every turn it:
      1. Retrieves the full prior history from Lakebase (its 'context'),
      2. Runs the OpenAI Agents SDK Agent (backed by DatabricksOpenAI) on the
         retrieved history + the new message,
      3. Persists BOTH the user's message and the agent's reply to Lakebase.
    """

    def __init__(self, session_id, memory, agent):
        self.session_id = session_id
        self.memory = memory
        self.agent = agent

    async def reply(self, user_text):
        # 1) Retrieve prior turns from Postgres — this is the agent's memory.
        history = self.memory.get_history(self.session_id)

        # 2) LLM call with the retrieved history as context.
        if history:
            convo = "\n".join(f"{t['role']}: {t['content']}" for t in history)
            prompt = f"Conversation so far:\n{convo}\n\nuser: {user_text}"
        else:
            prompt = user_text
        result = await Runner.run(self.agent, prompt)
        response = result.final_output

        # 3) Persist both sides of the conversation to Postgres.
        next_turn = (history[-1]["turn"] + 1) if history else 1
        self.memory.save_turn(self.session_id, next_turn, "user", user_text)
        self.memory.save_turn(self.session_id, next_turn + 1, "agent", response)
        return response


print("DemoAgent (DatabricksOpenAI-backed) ready.")

DemoAgent (DatabricksOpenAI-backed) ready.


In [0]:
# --- Session 1: a first conversation ---
# Same flow as the other demos — replies from the LLM, every turn saved
# to Lakebase.
SESSION_ID = "dboai-demo-session-1"

agent = DemoAgent(SESSION_ID, memory, llm_agent)

for user_message in ["Hi, I'm Alan", "I like tea."]:
    print(f"User : {user_message}")
    print(f"Agent: {await agent.reply(user_message)}\n")

print("What the agent just stored in Postgres:")
for t in memory.get_history(SESSION_ID):
    print(f"  turn {t['turn']:>2} [{t['role']:>5}] {t['content']}")

User : Hi, I'm Alan
Agent: Nice to meet you, Alan. How can I assist you today?

User : I like tea.
Agent: That's great, Alan, we can discuss tea if you'd like, or is there something else on your mind that I can help with?

What the agent just stored in Postgres:
  turn  1 [ user] Hi, I'm Alan
  turn  2 [agent] Nice to meet you, Alan. How can I assist you today?
  turn  3 [ user] I like tea.
  turn  4 [agent] That's great, Alan, we can discuss tea if you'd like, or is there something else on your mind that I can help with?


In [0]:
# --- Simulate a total agent restart ---
# Destroy the in-process agent. The brand-new instance has ZERO in-memory
# state — only Lakebase holds the memory. Same punchline, now on the
# Databricks-native client.
del agent

agent2 = DemoAgent(SESSION_ID, memory, llm_agent)  # same session, fresh "process"

user_message = "What's my name? What do I like?"
print(f"User : {user_message}")
print(f"Agent: {await agent2.reply(user_message)}")
print()
print("The new agent instance answered from Lakebase memory — the LLM alone would have forgotten.")

# Show the raw table: everything the agent has ever stored.
conn = get_connection()
rows = conn.run(
    "SELECT session_id, turn, role, content, created_at "
    "FROM agent_session_memory_databricks ORDER BY session_id, turn"
)
conn.close()
print("\nRaw table contents (agent_session_memory_databricks):")
for row in rows:
    print(f"  {row[0]} | turn {row[1]:>2} | {row[2]:>5} | {row[3]} | {row[4]}")

User : What's my name? What do I like?
Agent: Your name is Alan, and you like tea.

The new agent instance answered from Lakebase memory — the LLM alone would have forgotten.

Raw table contents (agent_session_memory_databricks):
  dboai-demo-session-1 | turn  1 |  user | Hi, I'm Alan | 2026-10-27 19:47:58.152626+00:00
  dboai-demo-session-1 | turn  2 | agent | Nice to meet you, Alan. How can I assist you today? | 2026-10-27 19:47:58.431130+00:00
  dboai-demo-session-1 | turn  3 |  user | I like tea. | 2026-10-27 19:48:00.160195+00:00
  dboai-demo-session-1 | turn  4 | agent | That's great, Alan, we can discuss tea if you'd like, or is there something else on your mind that I can help with? | 2026-10-27 19:48:00.423615+00:00
  dboai-demo-session-1 | turn  5 |  user | What's my name? What do I like? | 2026-10-27 19:48:02.390401+00:00
  dboai-demo-session-1 | turn  6 | agent | Your name is Alan, and you like tea. | 2026-10-27 19:48:02.662290+00:00


## Takeaways

### Two ways to reach Unity Gateway from the OpenAI ecosystem

| | Manual OpenAI client (previous notebook) | `DatabricksOpenAI` (this notebook) |
| --- | --- | --- |
| Auth | Mint bearer from `w.config.authenticate()`, refresh ~hourly | Unified auth — zero auth code, auto-refreshed |
| Routing | Hand-build `base_url` … `/ai-gateway/mlflow/v1` | One flag: `use_ai_gateway=True` |
| Default target | (whatever you point it at) | ⚠️ classic `/serving-endpoints` — **the flag is required** for `system.ai.*` names |
| Extras | none | UC function tools, Vector Search retriever, MCP server toolkit |

Both notebooks prove the same core point, though:

* **The memory loop is the story.** The LLM is stateless between calls — every "memory" the agent displayed came from history retrieved out of Lakebase and passed back in as context.
* **Framework- and client-agnostic memory.** `SessionMemory` didn't change across the deterministic, manual-OpenAI, and DatabricksOpenAI versions — the memory layer doesn't care what sits on top.
* **Governed model access.** Same UC model service (`system.ai.llama-4-maverick`), same usage tracking (`system.ai_gateway.usage`), same one-line swap to a bigger model.

### Where to take this next

* **Databricks-native tools** — the `databricks-openai` package ships `UCFunctionToolkit` (call UC functions as agent tools), `VectorSearchRetrieverTool`, and `McpServerToolkit`; giving this agent a "remember" tool that writes to Lakebase is a natural next demo.
* **Port to the Streamlit app** — in the deployed app, `DatabricksOpenAI(use_ai_gateway=True)` needs no token handling at all (the app's service principal authenticates), making the app code even simpler than the notebook version.
* **Replies are non-deterministic** — wording varies run to run; the *facts* come from stored memory. Demo 1 (deterministic) remains the live-session fallback.